## Experiment Tracking machine learning pipeline

### By:
[Gabriel Múnera](https://github.com/gamug)

### Date:
2026-08-21

### Description:

Experiment-track the parameters, metrics and model artifact for the diamond price model selected and trained in
`notebooks/5-models/03-gmg-first_model-2026_08_18.ipynb` (`HistGradientBoostingRegressor` inside a
`TransformedTargetRegressor`, over the preprocessing pipeline from
`notebooks/4-feat_eng/01-gmg-basic-feature-engineering-pipeline-2026_08_18.ipynb`), using **MLflow**.

Tracking the run (hyperparameters, MAE/RMSE/MAPE/R², the fitted pipeline) makes the model comparable and
reproducible as the project iterates on more model families or feature sets later.

## 📚 Import  libraries

In [1]:
# base libraries for data science
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from joblib import dump
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, StandardScaler

## 💾 Load data

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

dataset = pd.read_parquet(DATA_DIR / "04_feature/diamantes_clean.parquet", engine="pyarrow")
dataset.shape

(18977, 10)

## 👷 Data preparation

The columns that will be used are:

`['carat', 'depth', 'table', 'x', 'y', 'z', 'cut', 'color', 'clarity']` to predict `price`.

In [ ]:
numeric_features = ["carat", "depth", "table"]
volume_source_features = ["x", "y", "z"]
categorical_ordinal_features = ["cut", "color", "clarity"]
target = "price"

selected_features = numeric_features + volume_source_features + categorical_ordinal_features
dataset_features = dataset[[*selected_features, target]]

## Convert data types

In [4]:
dataset_features[numeric_features + volume_source_features] = dataset_features[
    numeric_features + volume_source_features
].astype("float32")
dataset_features[categorical_ordinal_features] = dataset_features[
    categorical_ordinal_features
].astype("category")
dataset_features[target] = dataset_features[target].astype("int64")

## 👨‍🏭 Feature Engineering

In [ ]:
ordinal_categories = {
    "cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    "color": ["J", "I", "H", "G", "F", "E", "D"],
    "clarity": ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
}


def compute_volume(X: np.ndarray) -> np.ndarray:
    x, y, z = X[:, 0], X[:, 1], X[:, 2]
    return (x * y * z).reshape(-1, 1)


def volume_feature_names_out(
    transformer: FunctionTransformer, input_features: list[str]
) -> np.ndarray:
    """Named (picklable) replacement for a lambda: FunctionTransformer always outputs 1 column, 'volume'."""
    return np.array(["volume"])


volume_transformer = FunctionTransformer(compute_volume, feature_names_out=volume_feature_names_out)

numeric_pipe = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
volume_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("volume", volume_transformer),
        ("scaler", StandardScaler()),
    ]
)
categorical_ord_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=[ordinal_categories[c] for c in categorical_ordinal_features]
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("volume", volume_pipe, volume_source_features),
        ("cat_ordinal", categorical_ord_pipe, categorical_ordinal_features),
    ]
)

## Train / Test split

In [6]:
X_features = dataset_features[selected_features]
Y_target = dataset_features[target]

clarity_cut_key = (
    dataset_features["clarity"].astype(str) + "_" + dataset_features["cut"].astype(str)
)

x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, stratify=clarity_cut_key, test_size=0.2, random_state=42
)
x_train.shape, x_test.shape

((15181, 9), (3796, 9))

### Create pipeline

In [7]:
data_model_pipeline = TransformedTargetRegressor(
    regressor=Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", HistGradientBoostingRegressor(random_state=42)),
        ]
    ),
    func=np.log,
    inverse_func=np.exp,
)

## Hyperparameter tunning

Same search space validated in `03-...first_model.ipynb`.

### Hist Gradient Boosting

In [8]:
score = "neg_mean_absolute_percentage_error"

hyperparameters = {
    "regressor__model__max_iter": [200, 300, 400],
    "regressor__model__max_depth": [6, 10, None],
    "regressor__model__learning_rate": [0.03, 0.05, 0.1],
    "regressor__model__l2_regularization": [0.0, 1.0],
}

grid_search = GridSearchCV(
    data_model_pipeline,
    hyperparameters,
    cv=5,
    scoring=score,
    n_jobs=-1,
)
grid_search.fit(x_train, y_train)
grid_search.best_params_

{'regressor__model__l2_regularization': 0.0,
 'regressor__model__learning_rate': 0.05,
 'regressor__model__max_depth': 10,
 'regressor__model__max_iter': 300}

In [9]:
best_data_model_pipeline = grid_search.best_estimator_
type(best_data_model_pipeline)

sklearn.compose._target.TransformedTargetRegressor

### Evaluation

In [10]:
def regression_metrics(y_true, y_pred) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAPE_%": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "R2": r2_score(y_true, y_pred),
    }


y_pred = best_data_model_pipeline.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2 = r2_score(y_test, y_pred)
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"R2:   {r2:.4f}")

MAE:  559.35
RMSE: 856.33
MAPE: 6.84%
R2:   0.9518


## Track Experiment with MLflow

MLflow 3.x puts the plain filesystem store (`./mlruns`) in maintenance mode, so a local SQLite database is used
as the tracking backend instead (still fully local, no server to run).

In [11]:
mlflow_db_path = Path.cwd().resolve().parents[1] / "mlruns.db"
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db_path}")

exp = mlflow.set_experiment(experiment_name="diamantes_price_models")
with mlflow.start_run(run_name="hist_gradient_boosting-v1") as run:
    # infer the model signature
    signature = infer_signature(x_train, best_data_model_pipeline.predict(x_train))

    # log the tuned hyperparameters (strip the pipeline-step prefixes for readability)
    logged_params = {k.split("__")[-1]: v for k, v in grid_search.best_params_.items()}
    mlflow.log_params(logged_params)
    mlflow.log_param("model_family", "HistGradientBoostingRegressor")
    mlflow.log_param("target_transform", "log")
    mlflow.log_param("n_train_rows", len(x_train))
    mlflow.log_param("n_test_rows", len(x_test))

    # log the evaluation metrics
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)
    mlflow.log_metric("R2", r2)
    mlflow.log_metric("CV_MAPE", -grid_search.best_score_ * 100)

    # log the fitted pipeline as an MLflow model.
    # MLflow 3.x defaults to `skops` serialization, which refuses to (de)serialize the notebook-local
    # `compute_volume` / `volume_feature_names_out` functions used by the FunctionTransformer as "untrusted
    # types" - use `cloudpickle` instead, which can serialize arbitrary Python callables safely within a
    # trusted, single-team project like this one.
    mlflow.sklearn.log_model(
        sk_model=best_data_model_pipeline,
        name="model",
        signature=signature,
        input_example=x_train.head(),
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

    run_id = run.info.run_id
    print(f"MLflow run id: {run_id}")

2026/08/21 15:44:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/08/21 15:45:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MLflow run id: 4d2a312b4e484d4e9a2a61f2f6e13773


In [12]:
# quick look at what was logged for this run
logged_run = mlflow.get_run(run_id)
pd.Series(logged_run.data.metrics)

MAE        559.354729
RMSE       856.327781
MAPE         6.837331
R2           0.951792
CV_MAPE      7.100873
dtype: float64

## Save the model

In [13]:
MODEL_DIR = DATA_DIR / "06_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
# save with MLflow's own format (keeps environment / signature metadata alongside the model)
mlflow_model_path = MODEL_DIR / "diamantes_price-hist_gradient_boosting-mlflow"
if mlflow_model_path.exists():
    import shutil

    shutil.rmtree(mlflow_model_path)

mlflow.sklearn.save_model(
    sk_model=best_data_model_pipeline,
    path=mlflow_model_path,
    signature=signature,
    input_example=x_train.head(),
    serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
)

# also keep the plain joblib artifact in sync (same format used by 03-...first_model.ipynb)
dump(
    best_data_model_pipeline,
    MODEL_DIR / "diamantes_price-hist_gradient_boosting-v1.joblib",
    protocol=5,
)
sorted(p.name for p in MODEL_DIR.iterdir() if p.name != ".gitkeep")

2026/08/21 15:45:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/08/21 15:45:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


['diamantes_price-hist_gradient_boosting-mlflow',
 'diamantes_price-hist_gradient_boosting-v1.joblib']

## Load Model to make predictions

In [15]:
load_model = mlflow.sklearn.load_model(model_uri=str(mlflow_model_path))

predictions = load_model.predict(x_test)
predictions[:8]

array([11384.74059944,  9663.30063233,  9255.00467717, 11940.54146152,
        5294.05475894,  8696.3074424 , 12459.38635544,  3905.63040824])

In [16]:
y_test.head(8).to_numpy()

array([11873,  9424,  9979, 13465,  5239,  8940, 14949,  4414])

## 📊 Analysis of Results

- The MLflow run (`run_id` logged at execution time, experiment `diamantes_price_models`) recorded the same
  metrics obtained in `03-...first_model.ipynb`: **MAE ≈ \$559.35, RMSE ≈ \$856.33, MAPE ≈ 6.84%, R² ≈ 0.952**
  on the test set, alongside `CV_MAPE ≈ 7.10%` from the grid search.
- Both the **MLflow model format** (`data/06_models/diamantes_price-hist_gradient_boosting-mlflow/`, with
  environment + signature metadata) and the **plain joblib artifact**
  (`data/06_models/diamantes_price-hist_gradient_boosting-v1.joblib`) are kept in sync with identical
  predictions, so either can be used downstream depending on whether MLflow tooling is available.
- **Serialization note**: MLflow 3.x defaults to the `skops` format for scikit-learn models, but it rejects the
  notebook-local `compute_volume` / `volume_feature_names_out` functions used inside the `FunctionTransformer`
  as "untrusted types". Switching to `serialization_format="cloudpickle"` resolves this and is an acceptable
  trade-off in this trusted, single-team academic project (cloudpickle can execute arbitrary code on load, so it
  would need revisiting before serving this artifact from an untrusted source).
- Loading the saved MLflow model and predicting on 8 held-out rows reproduces predictions consistent with the
  ~6.8% average error already established (e.g. \$11,384.74 vs \$11,873 actual; \$3,905.63 vs \$4,414 actual).

_(filled in after execution)_

## 🧑‍🔬 Recommendations:

1. **Use the MLflow run as the system of record** for this model's hyperparameters and metrics going forward —
   every future retrain (new data, new features, new algorithm) should log a new run to the
   `diamantes_price_models` experiment so models stay comparable over time.
2. **Keep `serialization_format="cloudpickle"`** as long as the `FunctionTransformer`-based `volume` feature
   uses a plain Python function; if the model is ever served outside this trusted environment, either migrate to
   a `skops`-compatible transform (e.g. a class-based transformer with a `feature_names_out` method deepchecks
   with skops's allow-list) or keep cloudpickle but restrict who can load the artifact.
3. **Register the model in the MLflow Model Registry** (`registered_model_name=...`) once a stakeholder is ready
   to treat this as more than an experiment — this notebook logs the run but stops short of registering it, to
   keep this pass focused on tracking.
4. **Point `mlflow ui --backend-store-uri sqlite:///mlruns.db`** (run from the repo root) at the tracking
   database to browse this and future runs interactively.

_(filled in after execution)_

## 📖 References

- EDA notebook: `notebooks/3-analysis/02-jrz-data_description_Manual-pandas-2024_10_24.ipynb`
- Feature engineering notebook: `notebooks/4-feat_eng/01-gmg-basic-feature-engineering-pipeline-2026_08_18.ipynb`
- First model notebook: `notebooks/5-models/03-gmg-first_model-2026_08_18.ipynb`
- <https://mlflow.org/docs/latest/tracking.html>
- <https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html>